# RDT-1B resident adapter (`--backend rdt1b`)

Serves RDT-1B through the RoboticsDiffusionTransformer deployment wrapper from its own resident Flask adapter, `control/vla_bridge/rdt1b_server.py`, through an ngrok HTTP tunnel.

| | |
|---|---|
| Lane | HTTP adapter (isolated Python 3.10 venv) |
| Port → tunnel | 3005 → ngrok HTTP |
| GPU runtime | A100 |
| Gated HF licences | none |
| Sign-offs before running | none |
| Default checkpoint | `robotics-diffusion-transformer/rdt-1b@eb09036` |
| Server log | `/content/rdt1b_server.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py`, `rdt1b_server.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## Tier C: RDT-1B resident adapter (Plan 14-11, `--backend rdt1b`)

RDT-1B has no LeRobot `PolicyServer` integration. It runs as its own resident Flask adapter,
`control/vla_bridge/rdt1b_server.py`, built like the Octo and OpenVLA adapters (see README): it loads once,
warms up, then serves. It does not use the gRPC PolicyServer lane.

The code comes from **github.com/thu-ml/RoboticsDiffusionTransformer**, cloned at the pinned commit
`cd79363` (MIT). It is not on PyPI. The adapter loads through the repo's own deployment wrapper,
`scripts.agilex_model.RoboticDiffusionTransformerModel`, and overrides only the two hooks the README
says to change for a new robot (`_format_joint_to_state` / `_unformat_action_to_joint`). The
checkpoint, the SigLIP vision encoder and the T5-XXL text encoder are each downloaded at a pinned
revision.

**Status:** written from upstream source and HF Hub metadata. These cells have **not yet been run
on Colab**.

Requirements:
- **GPU: an A100 runtime (Colab Pro).** RDT runs in bfloat16, which a T4 lacks, so
  `rdt1b_server.py` refuses to start there (14-RESEARCH Pitfall 4). Expect about 13 GB of VRAM:
  T5-XXL's encoder is ~9.5 GB in bf16, RDT-1B 2.5 GB, SigLIP 0.8 GB. Below 24 GB of VRAM (e.g. an L4),
  the adapter automatically offloads T5 encoder blocks 12-23 to `--offload-dir`, as upstream recommends.
- **Disk: about 55 GB free.** `google/t5-v1_1-xxl` ships only a 44.5 GB fp32 `pytorch_model.bin`.
  The first launch therefore takes a long time. `--no-text-encoder` with precomputed
  `--lang-embeddings` (upstream `scripts/encode_lang.py` output) skips that download entirely.
- **Dependencies:** RDT pins Python 3.10 and `torch==2.1.0` (no Python 3.12 wheels), so the adapter
  runs in its own venv. `rdt1b_server.py` refuses other versions of torch, transformers, diffusers,
  timm, accelerate and huggingface_hub.

Default checkpoint: the official `robotics-diffusion-transformer/rdt-1b@eb09036`. Its action space was
confirmed from RDT's own code and weights as a 128-D unified vector whose **joint-position** slots
take SO-101 directly: arm joints 0-4 in radians, gripper slot 10 in [0, 1]. **No IK.** There is no
SO-101 RDT fine-tune on the Hub, and the model card says RDT does not generalize to robots it was
not pre-trained on. So this run proves the pipeline, not the task. The official
`robotics-diffusion-transformer/rdt-170m` is a reviewed alternate.

In [ ]:
# RDT-1B only: the official install (README: python 3.10, torch 2.1.0 cu121, requirements.txt) in an
# isolated venv, at a pinned thu-ml/RoboticsDiffusionTransformer commit. Only the inference subset of
# requirements.txt is installed (deepspeed, wandb, h5py and imgaug are training/data only). flash-attn is
# skipped: the model code uses torch SDPA through timm, never flash_attn. huggingface_hub is pinned to
# 0.25.2 because diffusers 0.27.2 still imports cached_download (removed in 0.26). Clone, installs and
# import check are all verified by exit status.
import subprocess, sys
from pathlib import Path
RDT_COMMIT = 'cd79363a1387e8f81c7724d070ef7e45fd23150f'
RDT_DIR = '/content/RoboticsDiffusionTransformer'
RDT_VENV = '/content/rdt_venv'
RDT_PYTHON = f'{RDT_VENV}/bin/python'
if not Path(RDT_DIR).exists():
    subprocess.run(['git', 'clone', 'https://github.com/thu-ml/RoboticsDiffusionTransformer', RDT_DIR], check=True)
subprocess.run(['git', '-C', RDT_DIR, 'checkout', '--detach', RDT_COMMIT], check=True)
head = subprocess.run(['git', '-C', RDT_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True)
if head.stdout.strip() != RDT_COMMIT:
    raise RuntimeError(f'RDT checkout is at {head.stdout.strip()}, expected {RDT_COMMIT}')
if not Path(RDT_PYTHON).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.10', '--seed', RDT_VENV], check=True)
for packages in (
    ['--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.1.0', 'torchvision==0.16.0'],
    ['transformers==4.41.0', 'diffusers==0.27.2', 'timm==1.0.3', 'accelerate==0.30.1', 'sentencepiece==0.2.0',
     'protobuf==4.25.3', 'huggingface_hub==0.25.2', 'packaging==24.0', 'numpy==1.26.4', 'pyyaml',
     'opencv-python-headless==4.9.0.80', 'flask', 'pillow', 'requests'],
):
    install = subprocess.run(['uv', 'pip', 'install', '--python', RDT_PYTHON, *packages], capture_output=True, text=True)
    print(install.stdout[-2000:]); print(install.stderr[-2000:])
    if install.returncode != 0:
        raise RuntimeError(f'RDT venv install failed (exit {install.returncode})')
check = subprocess.run([RDT_PYTHON, '-c', (
    "import importlib.metadata as m, torch\n"
    "print({p: m.version(p) for p in ('torch', 'transformers', 'diffusers', 'timm', 'accelerate', 'huggingface_hub', 'numpy')})\n"
    "from scripts.agilex_model import RoboticDiffusionTransformerModel\n"
    "from models.multimodal_encoder.t5_encoder import T5Embedder\n"
    "assert torch.cuda.is_available(), 'no CUDA GPU'\n"
    "print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')\n"
    "assert torch.cuda.is_bf16_supported(), 'this GPU has no bf16 (T4?): switch the runtime to an A100'\n"
)], cwd=RDT_DIR, capture_output=True, text=True)
print(check.stdout); print(check.stderr[-2000:])
if check.returncode != 0:
    raise RuntimeError('RDT venv import check failed; see stderr above')

## Adapter source and launch notes

The next cell writes `control/vla_bridge/rdt1b_server.py` into `/content/rdt1b_server.py`, and the cell after it checks its sha256. The launch cell then starts it as a separate OS process.

It runs with the venv's Python.

On first start it downloads three pinned revisions, about 51 GB in total:
- the checkpoint's `config.json` and `pytorch_model.bin` (2.5 GB for RDT-1B)
- `google/siglip-so400m-patch14-384` (3.5 GB)
- `google/t5-v1_1-xxl` (44.5 GB), unless you pass `--no-text-encoder`

Before declaring the model ready, it checks that the cloned repo's unified layout is the reviewed one
and that every weight key matches. Then it runs one warm-up inference. `/health` only answers after
that. It reports `action_space`, `state_indices`, `control_hz`, `chunk_step` and the T5 offload folder.
The log at `/content/rdt1b_server.log` records the GPU, whether T5 is offloaded, and the **peak VRAM**
after warm-up. That line is the Pitfall 4 evidence: an OOM shows up there as
`torch.cuda.OutOfMemoryError`, or as the process dying during load.

The server returns the chunk step about one client control period ahead. By default that is
`--client-control-hz 2.0`, matching `run_vla_episode.py`'s default `--control-hz`. If you run the
client at another rate, pass the same value in the launch cell's command.

To use the alternate, add `--checkpoint robotics-diffusion-transformer/rdt-170m` to the launch
cell's command, and pass the **same** `--checkpoint` to `run_vla_episode.py`.

In [ ]:
%%writefile /content/rdt1b_server.py
"""Resident RDT-1B HTTP adapter (Tier C, Plan 14-11, VLAB-09). Run on Colab inside
the isolated RDT venv (see vla_bridge/notebooks/http_rdt1b.ipynb).

Same shape as `octo_server.py` / `openvla_server.py` / `fastfs_server.py`:
- the model loads ONCE
- one throwaway warmup inference runs before the socket opens
- `/health` reports ready
- a non-blocking `inference_lock` answers concurrent requests with 429
- `ValueError`/`KeyError`/`TypeError` on a request become 400
- a model failure (`RuntimeError`, e.g. a non-finite action) becomes 500,
  so the client holds position instead of parsing NaN JSON

The server stays model-format-only. It returns the checkpoint's unified-action
slots for SO-101 (radians and a [0, 1] gripper), and `backends/rdt1b.py` turns
them into a `JOINT_ORDER` target in this rig's units.

Verified against live upstream source (2026-10-01, execution discipline)
------------------------------------------------------------------------
github.com/thu-ml/RoboticsDiffusionTransformer @ cd79363 (main HEAD, MIT). It is
NOT on PyPI. Files read: `README.md`, `requirements.txt`, `configs/base.yaml`,
`configs/state_vec.py`, `configs/dataset_control_freq.json`,
`scripts/agilex_model.py`, `scripts/agilex_inference.py`,
`scripts/encode_lang.py`, `models/rdt_runner.py`, `models/hub_mixin.py`,
`models/rdt/blocks.py`, `models/multimodal_encoder/t5_encoder.py`,
`models/multimodal_encoder/siglip_encoder.py`, `data/hdf5_vla_dataset.py`,
`train/train.py`, `train/sample.py`, `docs/test_6drot.py`.

- Unified action space (`configs/state_vec.py`, `STATE_VEC_LEN = 128`):
  `[0, 10)` right-arm joint positions, `10` right gripper open, `[30, 33)` EEF
  position, `[33, 39)` EEF 6-D rotation, `[50, 60)` left-arm joints, ...
  The README says a single-arm robot fills the RIGHT-arm part, physical
  quantities stay unnormalized in SI units (radians), and only the gripper is
  min-max normalized to [0, 1]. State and action share the same slots: training
  uses the state indicator as the action mask (`train/train.py`, `train/sample.py`),
  and `data/hdf5_vla_dataset.py` fills the ABSOLUTE target qpos into the same
  slots as the state. So SO-101's 5 arm joints map directly to slots 0-4 and
  its gripper to slot 10: a **joint-position** action space, no IK.
  `_check_state_vec_layout()` re-reads the mapping from the cloned repo at startup.
- Checkpoint, read from the weights themselves (HTTP range reads of
  `pytorch_model.bin`'s pickle, no full download): `rdt-1b` @ eb09036 has
  1,228,319,872 bf16 parameters. The final layer outputs 128 values, the state
  adaptor takes 256 (128 state + 128 mask), the image condition is 4374 = 2
  history x 3 cameras x 729 SigLIP patches, and the language input is 4096
  (T5-XXL). `rdt-170m` @ 8aa386c uses the same 128-D space (166M parameters).
- Loading: the official wrapper `scripts.agilex_model.RoboticDiffusionTransformerModel`.
  The README says to override `_format_joint_to_state()` and
  `_unformat_action_to_joint()` for a new robot; `_wrapper_class()` does exactly
  that. With a directory, `get_policy()` calls `RDTRunner.from_pretrained(dir)`
  (the README's documented path). `huggingface_hub==0.25.2`'s mixin injects
  `config.json` as `config` and loads `pytorch_model.bin` with `strict=False`, so
  `_verify_weights()` re-checks that no key is missing or unexpected.
- Inference (`step()`): 6 images `[ext, right_wrist, left_wrist] x [t-1, t]`.
  `None` becomes a background image, as `agilex_inference.py` does for the first
  step. The proprio state goes in as `(1, n)`, `ctrl_freqs` is the profile's
  control rate, and the output is `(1, 64, n)` absolute targets.
- Language: T5-XXL (`google/t5-v1_1-xxl`, 4096-D). The official deployment
  precomputes embeddings because of VRAM. `T5Embedder(use_offload_folder=...)`
  puts encoder blocks 12-23 on disk (`scripts/encode_lang.py`: "if your GPU VRAM
  is less than 24GB, it is recommended to enable offloading"). This adapter
  encodes live with `--offload-dir` (Pitfall 4), or serves only precomputed
  embeddings (`--lang-embeddings`, `--no-text-encoder`). Encoding mirrors
  `encode_lang.py` and is cached per instruction.
- Pins: Python 3.10, `torch==2.1.0`, `transformers==4.41.0`, `diffusers==0.27.2`,
  `timm==1.0.3`, `accelerate==0.30.1` (README / requirements.txt), plus
  `huggingface_hub==0.25.2`, because diffusers 0.27.2 imports `cached_download`,
  which huggingface_hub removed in 0.26. `main()` refuses other versions.

Checkpoints (HF API, read 2026-10-01; see `CHECKPOINT_PROFILES`)
----------------------------------------------------------------
- Default: `robotics-diffusion-transformer/rdt-1b` @ eb09036 (MIT, official).
  No RDT fine-tune for SO-101 (or SO-100 / Koch) exists on the Hub. The model card
  says RDT "cannot yet generalize to new robot platforms (not seen in the
  pre-training datasets)", so this is out of distribution until fine-tuned.
- Alternate: `robotics-diffusion-transformer/rdt-170m` @ 8aa386c (MIT, official).

Known limitations (documented, not hidden)
------------------------------------------
- The SO-101 slot profile follows the README's own guidance (right-arm slots, SI
  units, gripper = percent / 100, 0 % = closed). The base checkpoint never saw
  SO-101 joint conventions, so a fine-tune is required for meaningful motion.
- The camera order is ext = `overhead_left`, right wrist = `wrist`, and left
  wrist = background (single arm). `--ext-image-key` overrides ext.
- RDT predicts 64 steps at the profile's control rate (30 Hz). The server returns
  the step closest to one client control period ahead (`--client-control-hz`,
  default 2 Hz = `run_vla_episode.py`'s default -> step 14), not step 0, which
  would be only 1/30 s of motion per loop.
- The server keeps the 2-frame image history. It resets on a new instruction, on
  `"reset": true`, or after `HISTORY_RESET_S` seconds idle.
- bfloat16 only (the wrapper's dtype): `main()` refuses GPUs without native bf16
  (T4). T5-XXL alone is ~9.5 GB in bf16, plus a 44.5 GB fp32 download.

Colab launch: see vla_bridge/notebooks/http_rdt1b.ipynb.
"""
import argparse
import base64
import collections
import math
import os
import sys
import threading
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

DEFAULT_CHECKPOINT = "robotics-diffusion-transformer/rdt-1b"
DEFAULT_LOCAL_DIR = "/content/rdt_checkpoints"
DEFAULT_RDT_REPO = "/content/RoboticsDiffusionTransformer"
DEFAULT_OFFLOAD_DIR = "/content/rdt_t5_offload"
DEFAULT_PORT = 3005  # 3000 FastFS, 3001 OpenVLA-OFT, 3002 Wall-X, 3003 OpenVLA, 3004 Octo
DEFAULT_INSTRUCTION = "pick up the cube"
DEFAULT_CLIENT_CONTROL_HZ = 2.0  # run_vla_episode.py --control-hz default
RDT_COMMIT = "cd79363a1387e8f81c7724d070ef7e45fd23150f"  # github.com/thu-ml/RoboticsDiffusionTransformer main HEAD

# T-14-06: the only encoders RDT may load (HF API shas, read 2026-10-01).
T5_REPO = "google/t5-v1_1-xxl"
T5_REVISION = "3db67ab1af984cf10548a73467f0e5bca2aaaeb2"
T5_FILES = ("config.json", "pytorch_model.bin", "spiece.model", "special_tokens_map.json", "tokenizer_config.json")
SIGLIP_REPO = "google/siglip-so400m-patch14-384"
SIGLIP_REVISION = "9fdffc58afc957d1a03a25b10dba0329ab15c2a3"
SIGLIP_FILES = ("config.json", "preprocessor_config.json", "model.safetensors")
CHECKPOINT_FILES = ("config.json", "pytorch_model.bin")
OFFLOAD_VRAM_THRESHOLD_GIB = 24.0  # scripts/encode_lang.py: offload below 24 GB

# action_contract.JOINT_ORDER (pinned equal by test_server_joint_order_matches_the_action_contract);
# this file runs standalone on Colab, so it does not import vla_bridge.
SO101_JOINTS = ("shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll", "gripper")
# configs/state_vec.py @ RDT_COMMIT: right_arm_joint_{0..4}_pos -> 0..4, right_gripper_open -> 10.
SO101_JOINT_SLOTS = (0, 1, 2, 3, 4, 10)
STATE_VEC_LEN = 128

# Reviewed checkpoints. `backend_configs/rdt1b.json["checkpoints"]` must agree
# (pinned by test_client_and_server_checkpoint_profiles_agree).
_SO101_JOINT_PROFILE = dict(action_space="joint_position", state_indices=list(SO101_JOINT_SLOTS),
                            joint_units="rad", gripper_norm_to_percent=[0.0, 100.0], control_hz=30)
CHECKPOINT_PROFILES = {
    "robotics-diffusion-transformer/rdt-1b": dict(
        revision="eb09036cc64ca4945051acbd1bd581d30a1d7711", **_SO101_JOINT_PROFILE),
    "robotics-diffusion-transformer/rdt-170m": dict(
        revision="8aa386cac3bbfd9540676c75b3d767cc7f88a10a", **_SO101_JOINT_PROFILE),
}

MAX_IMAGE_SIDE = 1024
MAX_INSTRUCTION_CHARS = 1000
MAX_LIVE_EMBEDDINGS = 64
IMAGE_KEYS = ("overhead_left", "overhead_right", "wrist")
HISTORY_RESET_S = 10.0
PINNED_DEPENDENCIES = {"torch": {"2.1.0"}, "transformers": {"4.41.0"}, "diffusers": {"0.27.2"},
                       "timm": {"1.0.3"}, "accelerate": {"0.30.1"}, "huggingface_hub": {"0.25.2"}}


def _download_pinned(repo_id, revision, local_dir, patterns):
    """Materialise exactly `patterns` of `repo_id` at the pinned `revision` as a
    real directory (the upstream loaders would otherwise resolve `main`)."""
    from huggingface_hub import snapshot_download

    target = Path(local_dir) / repo_id.replace("/", "--") / revision
    return snapshot_download(repo_id=repo_id, revision=revision, local_dir=str(target),
                             allow_patterns=list(patterns))


def _add_repo_to_path(rdt_repo):
    """The upstream code imports `configs.*` / `models.*` / `scripts.*` from its repo root."""
    root = str(Path(rdt_repo).resolve())
    if not (Path(root) / "scripts" / "agilex_model.py").is_file():
        raise RuntimeError(f"{root} is not a RoboticsDiffusionTransformer checkout (no scripts/agilex_model.py)")
    if root not in sys.path:
        sys.path.insert(0, root)


def _check_state_vec_layout(rdt_repo):
    """Refuse to run if the cloned repo's unified layout is not the one this
    adapter (and backends/rdt1b.py) was reviewed against."""
    _add_repo_to_path(rdt_repo)
    from configs.state_vec import STATE_VEC_IDX_MAPPING, STATE_VEC_LEN as upstream_len

    names = [f"right_arm_joint_{i}_pos" for i in range(5)] + ["right_gripper_open"]
    found = tuple(STATE_VEC_IDX_MAPPING.get(name) for name in names)
    if found != SO101_JOINT_SLOTS or upstream_len != STATE_VEC_LEN:
        raise SystemExit(f"{rdt_repo}/configs/state_vec.py maps {dict(zip(names, found))} (len {upstream_len}); "
                         f"expected {SO101_JOINT_SLOTS} of {STATE_VEC_LEN}. Check out {RDT_COMMIT}.")


def _wrapper_class(base, state_indices):
    """The README's documented robot adaptation: override `_format_joint_to_state()`
    and `_unformat_action_to_joint()` of `RoboticDiffusionTransformerModel`.
    Values arrive already in model units (radians, gripper 0-1)."""
    indices = list(state_indices)

    class SO101RoboticDiffusionTransformerModel(base):
        def _format_joint_to_state(self, joints):
            import torch

            batch, steps, _ = joints.shape
            dim = self.args["model"]["state_token_dim"]
            state = torch.zeros((batch, steps, dim), device=joints.device, dtype=joints.dtype)
            state[:, :, indices] = joints
            state_elem_mask = torch.zeros((batch, dim), device=joints.device, dtype=joints.dtype)
            state_elem_mask[:, indices] = 1
            return state, state_elem_mask

        def _unformat_action_to_joint(self, action):
            return action[:, :, indices]

    return SO101RoboticDiffusionTransformerModel


def _verify_weights(policy, weights_file):
    """`CompatiblePyTorchModelHubMixin` loads with strict=False; a silent key
    mismatch would leave random weights, so re-check it strictly here."""
    import torch

    state_dict = torch.load(str(weights_file), map_location="cpu", weights_only=True)
    result = policy.load_state_dict(state_dict, strict=False)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f"{weights_file} does not match RDTRunner: missing {result.missing_keys[:5]}, "
                           f"unexpected {result.unexpected_keys[:5]}")


def _load_base_config(rdt_repo):
    import yaml

    with open(Path(rdt_repo) / "configs" / "base.yaml") as handle:
        return yaml.safe_load(handle)


def _load_policy(rdt_repo, model_dir, vision_dir, control_hz, state_indices, device):
    """The official load (README "Deployment on Real-Robots" / scripts/agilex_inference.py)."""
    import torch

    _add_repo_to_path(rdt_repo)
    from scripts.agilex_model import RoboticDiffusionTransformerModel

    model = _wrapper_class(RoboticDiffusionTransformerModel, state_indices)(
        _load_base_config(rdt_repo), device=device, dtype=torch.bfloat16, control_frequency=control_hz,
        pretrained=str(model_dir), pretrained_vision_encoder_name_or_path=str(vision_dir))
    _verify_weights(model.policy, Path(model_dir) / "pytorch_model.bin")
    return model


def _load_text_encoder(rdt_repo, t5_dir, offload_dir, max_length, device):
    """Upstream `T5Embedder`, with the offload folder threaded into its
    `use_offload_folder` (encoder blocks 12-23 on disk). Its `available_models`
    allow-list holds only the repo id, so the pinned local copy is added to it."""
    _add_repo_to_path(rdt_repo)
    from models.multimodal_encoder.t5_encoder import T5Embedder

    t5_dir = str(t5_dir)
    if t5_dir not in T5Embedder.available_models:
        T5Embedder.available_models = [*T5Embedder.available_models, t5_dir]
    if offload_dir:
        os.makedirs(offload_dir, exist_ok=True)
    return T5Embedder(device=device, from_pretrained=t5_dir, model_max_length=max_length,
                      use_offload_folder=offload_dir or None)


def _encode_instruction(embedder, instruction, device):
    """Exactly `scripts/encode_lang.py`: (1, L, 4096) last hidden state, on CPU."""
    import torch

    tokens = embedder.tokenizer(instruction, return_tensors="pt", padding="longest",
                                truncation=True)["input_ids"].to(device)
    tokens = tokens.view(1, -1)
    with torch.no_grad():
        return embedder.model(tokens).last_hidden_state.detach().cpu()


def _load_lang_embedding(path):
    """A `scripts/encode_lang.py` output: {"name", "instruction", "embeddings": (1, L, 4096)}."""
    import torch

    saved = torch.load(str(path), map_location="cpu")
    instruction, embeddings = saved.get("instruction"), saved.get("embeddings")
    if not isinstance(instruction, str) or getattr(embeddings, "ndim", None) != 3 or embeddings.shape[0] != 1:
        raise ValueError(f"{path} is not an encode_lang.py embedding file")
    return instruction, embeddings


def _step(model, proprio, images, text_embeds):
    """`RoboticDiffusionTransformerModel.step()`; returns (1, horizon, n) numpy."""
    import torch

    pil_images = [Image.fromarray(image) if image is not None else None for image in images]
    actions = model.step(proprio=torch.tensor([proprio], dtype=torch.float32), images=pil_images,
                         text_embeds=text_embeds)
    return actions.float().cpu().numpy()


def chunk_step_for(control_hz, client_control_hz, horizon):
    """Index of the chunk step about one client control period ahead."""
    if client_control_hz <= 0:
        raise ValueError("client control rate must be positive")
    return int(min(max(round(control_hz / client_control_hz) - 1, 0), horizon - 1))


def joint_state_to_model(joint_state, profile):
    """Rig units (degrees, gripper percent) -> the profile's slot units."""
    if not isinstance(joint_state, dict):
        raise TypeError("joint_state must be an object")
    values = []
    for name in SO101_JOINTS:
        value = joint_state[name]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            raise ValueError(f"joint_state[{name!r}] must be a finite number")
        values.append(float(value))
    arm = [math.radians(v) if profile["joint_units"] == "rad" else v for v in values[:5]]
    lo, hi = (float(v) for v in profile["gripper_norm_to_percent"])
    return arm + [(values[5] - lo) / (hi - lo)]


class RDT1BModel:
    def __init__(self, checkpoint=DEFAULT_CHECKPOINT, revision=None, rdt_repo=DEFAULT_RDT_REPO,
                 local_dir=DEFAULT_LOCAL_DIR, offload_dir=None, ext_image_key="overhead_left",
                 client_control_hz=DEFAULT_CLIENT_CONTROL_HZ, lang_embeddings=(), text_encoder=True,
                 device="cuda"):
        profile = CHECKPOINT_PROFILES.get(checkpoint)
        if profile is None:
            raise ValueError(f"{checkpoint} has no reviewed slot profile (known: {sorted(CHECKPOINT_PROFILES)})")
        if profile["action_space"] != "joint_position":
            raise RuntimeError(f"{checkpoint}: only joint_position profiles are served; an EEF profile needs "
                               "the current EEF pose as state, which the wire contract does not carry")
        if ext_image_key not in IMAGE_KEYS:
            raise ValueError(f"ext image key must be one of {IMAGE_KEYS}")
        revision = revision or profile["revision"]
        model_dir = _download_pinned(checkpoint, revision, local_dir, CHECKPOINT_FILES)
        vision_dir = _download_pinned(SIGLIP_REPO, SIGLIP_REVISION, local_dir, SIGLIP_FILES)
        self.model = _load_policy(rdt_repo, model_dir, vision_dir, profile["control_hz"],
                                  profile["state_indices"], device)
        common = self.model.args["common"]
        if int(common["img_history_size"]) != 2 or int(common["num_cameras"]) != 3:
            raise RuntimeError(f"base.yaml expects {common['img_history_size']} frames x {common['num_cameras']} "
                               "cameras; this adapter feeds 2 x 3")
        self.horizon = int(common["action_chunk_size"])
        self.chunk_step = chunk_step_for(profile["control_hz"], client_control_hz, self.horizon)
        self.profile = profile
        self.device = device
        self._precomputed = {}
        for path in lang_embeddings:
            instruction, embeddings = _load_lang_embedding(path)
            self._precomputed[instruction] = embeddings
        self._live = collections.OrderedDict()
        self.text_encoder = None
        if text_encoder:
            t5_dir = _download_pinned(T5_REPO, T5_REVISION, local_dir, T5_FILES)
            self.text_encoder = _load_text_encoder(rdt_repo, t5_dir, offload_dir,
                                                   self.model.args["dataset"]["tokenizer_max_length"], device)
        elif not self._precomputed:
            raise ValueError("without the live text encoder, at least one --lang-embeddings file is required")
        self.image_sources = (ext_image_key, "wrist", None)  # [ext, right_wrist, left_wrist]
        self.required_image_keys = tuple(dict.fromkeys(k for k in self.image_sources if k))
        self._history = collections.deque(maxlen=2)
        self._last_instruction = None
        self._last_time = None
        self.model_version = f"{checkpoint}@{revision[:12]}"
        self.metadata = dict(
            checkpoint=checkpoint, revision=revision, rdt_commit=RDT_COMMIT,
            vision_encoder=f"{SIGLIP_REPO}@{SIGLIP_REVISION}",
            text_encoder=f"{T5_REPO}@{T5_REVISION}" if text_encoder else "precomputed only",
            t5_offload_dir=(offload_dir or None) if text_encoder else None,
            precomputed_instructions=sorted(self._precomputed),
            action_space=profile["action_space"], state_indices=list(profile["state_indices"]),
            joint_units=profile["joint_units"], gripper_norm_to_percent=list(profile["gripper_norm_to_percent"]),
            control_hz=profile["control_hz"], action_horizon=self.horizon, chunk_step=self.chunk_step,
            client_control_hz=client_control_hz, img_history_size=2,
            image_order=dict(ext=ext_image_key, right_wrist="wrist", left_wrist="background"),
            returns="chunk[chunk_step] of the profile's slots: 5 arm joints (rad), gripper_open (0-1)",
        )

    def reset(self):
        self._history.clear()
        self._last_instruction = None

    def _text_embeds(self, instruction):
        if instruction in self._precomputed:
            return self._precomputed[instruction]
        if instruction in self._live:
            self._live.move_to_end(instruction)
            return self._live[instruction]
        if self.text_encoder is None:
            raise ValueError(f"no precomputed embedding for {instruction!r} (known: {sorted(self._precomputed)})")
        embeddings = _encode_instruction(self.text_encoder, instruction, self.device)
        self._live[instruction] = embeddings
        while len(self._live) > MAX_LIVE_EMBEDDINGS:
            self._live.popitem(last=False)
        return embeddings

    def __call__(self, images, instruction, joint_state, reset=False):
        proprio = joint_state_to_model(joint_state, self.profile)  # 400 before touching history
        if len(instruction) > MAX_INSTRUCTION_CHARS:
            raise ValueError(f"instruction must be <= {MAX_INSTRUCTION_CHARS} characters")
        text_embeds = self._text_embeds(instruction)
        now = time.monotonic()
        stale = self._last_time is not None and now - self._last_time > HISTORY_RESET_S
        if reset or stale or instruction != self._last_instruction:
            self.reset()
        self._last_time = now
        self._last_instruction = instruction
        current = tuple(images[key] if key else None for key in self.image_sources)
        previous = self._history[-1] if self._history else (None, None, None)  # agilex: background at t-1
        self._history.append(current)
        chunk = np.asarray(_step(self.model, proprio, [*previous, *current], text_embeds), dtype=np.float64)
        width = len(self.profile["state_indices"])
        if chunk.ndim != 3 or chunk.shape[0] != 1 or chunk.shape[2] != width or chunk.shape[1] <= self.chunk_step:
            raise RuntimeError(f"model returned actions of shape {chunk.shape}, "
                               f"expected (1, {self.horizon}, {width})")
        values = chunk[0, self.chunk_step]
        if not np.all(np.isfinite(values)):
            raise RuntimeError(f"model returned a non-finite action {values.tolist()}")
        return [float(v) for v in values]


def _decode_rgb(b64_png):
    """base64 PNG (OpenCV BGR-encoded by the client) -> HxWx3 uint8 RGB."""
    if not isinstance(b64_png, str):
        raise TypeError("image must be a base64 string")
    raw = base64.b64decode(b64_png, validate=True)
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError("invalid PNG")
    if max(frame.shape[:2]) > MAX_IMAGE_SIDE:
        raise ValueError(f"image dimensions must be <= {MAX_IMAGE_SIDE}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()
    image_keys = tuple(getattr(infer, 'required_image_keys', ('overhead_left', 'wrist')))

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/action')
    def action():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='rdt service busy'), 429
        try:
            body = request.get_json(silent=True)
            if not isinstance(body, dict):
                raise ValueError('expected a JSON object body')
            images_in = body['images']
            if not isinstance(images_in, dict):
                raise TypeError('images must be an object')
            images = {key: _decode_rgb(images_in[key]) for key in image_keys}
            instruction = body['instruction']
            if not isinstance(instruction, str) or not instruction.strip():
                raise ValueError('instruction must be a non-empty string')
            joint_state = body['joint_state']  # RDT conditions on proprio: required
            reset = body.get('reset', False)
            if not isinstance(reset, bool):
                raise TypeError('reset must be a boolean')
            started = time.perf_counter()
            result = infer(images, instruction, joint_state, reset=reset)
            elapsed = (time.perf_counter() - started) * 1000
            return jsonify(action=[float(v) for v in result], model_version=infer.model_version,
                           inference_ms=elapsed)
        except (ValueError, KeyError, TypeError) as exc:
            return jsonify(error=str(exc)), 400
        except RuntimeError as exc:
            return jsonify(error=f'rdt inference failed: {exc}'), 500
        finally:
            inference_lock.release()
    return app


def _installed_versions():
    import importlib.metadata as metadata

    return {name: metadata.version(name).split("+")[0] for name in PINNED_DEPENDENCIES}


def _check_dependency_versions():
    versions = _installed_versions()
    bad = {name: version for name, version in versions.items() if version not in PINNED_DEPENDENCIES[name]}
    if bad:
        raise SystemExit(
            "RDT needs torch==2.1.0, transformers==4.41.0, diffusers==0.27.2, timm==1.0.3, accelerate==0.30.1, "
            f"huggingface_hub==0.25.2 (RDT README / requirements.txt); found {bad}. Use the isolated RDT venv."
        )


def _gpu_info():
    import torch

    if not torch.cuda.is_available():
        return dict(cuda=False, name=None, total_gib=0.0, bf16=False)
    props = torch.cuda.get_device_properties(0)
    return dict(cuda=True, name=props.name, total_gib=props.total_memory / 2**30,
                bf16=bool(torch.cuda.is_bf16_supported()))


def _peak_vram_gib():
    import torch

    return torch.cuda.max_memory_allocated() / 2**30


def resolve_offload_dir(mode, offload_dir, total_gib):
    """`auto` follows encode_lang.py's guidance: offload T5-XXL below 24 GB of VRAM."""
    if mode == "off":
        return None
    if mode == "on" or total_gib < OFFLOAD_VRAM_THRESHOLD_GIB:
        return offload_dir
    return None


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--checkpoint', default=DEFAULT_CHECKPOINT, choices=sorted(CHECKPOINT_PROFILES))
    parser.add_argument('--revision', default=None, help="pinned HF commit sha (defaults to the reviewed profile's)")
    parser.add_argument('--rdt-repo', default=DEFAULT_RDT_REPO, help=f'RoboticsDiffusionTransformer checkout ({RDT_COMMIT[:7]})')
    parser.add_argument('--local-dir', default=DEFAULT_LOCAL_DIR)
    parser.add_argument('--offload-dir', default=DEFAULT_OFFLOAD_DIR,
                        help='T5-XXL offload folder (T5Embedder use_offload_folder); needs ~5 GB of free disk')
    parser.add_argument('--t5-offload', choices=('auto', 'on', 'off'), default='auto',
                        help=f'auto = offload when the GPU has < {OFFLOAD_VRAM_THRESHOLD_GIB:g} GiB')
    parser.add_argument('--lang-embeddings', action='append', default=[],
                        help='scripts/encode_lang.py output (.pt); repeatable')
    parser.add_argument('--no-text-encoder', action='store_true', help='serve precomputed instructions only')
    parser.add_argument('--ext-image-key', default='overhead_left', choices=IMAGE_KEYS)
    parser.add_argument('--client-control-hz', type=float, default=DEFAULT_CLIENT_CONTROL_HZ,
                        help="run_vla_episode.py's --control-hz; picks the returned chunk step")
    parser.add_argument('--warmup-instruction', default=DEFAULT_INSTRUCTION)
    parser.add_argument('--port', type=int, default=DEFAULT_PORT)
    args = parser.parse_args()
    _check_dependency_versions()
    _check_state_vec_layout(args.rdt_repo)
    gpu = _gpu_info()
    if not gpu['cuda'] or not gpu['bf16']:
        raise SystemExit(f"RDT runs in bfloat16 and needs a GPU with native bf16 (A100/L4 class); found {gpu}. "
                         "A T4 cannot run it (14-RESEARCH Pitfall 4).")
    offload_dir = resolve_offload_dir(args.t5_offload, args.offload_dir, gpu['total_gib'])
    print(f"GPU {gpu['name']} {gpu['total_gib']:.1f} GiB; T5-XXL offload: {offload_dir or 'off'}", flush=True)
    model = RDT1BModel(args.checkpoint, revision=args.revision, rdt_repo=args.rdt_repo, local_dir=args.local_dir,
                       offload_dir=offload_dir, ext_image_key=args.ext_image_key,
                       client_control_hz=args.client_control_hz, lang_embeddings=args.lang_embeddings,
                       text_encoder=not args.no_text_encoder)
    warmup = args.warmup_instruction if not args.no_text_encoder else model.metadata['precomputed_instructions'][0]
    model({key: np.zeros((384, 384, 3), dtype=np.uint8) for key in model.required_image_keys}, warmup,
          {name: 0.0 for name in SO101_JOINTS})
    model.reset()  # the warmup frame must not become the first real request's history
    print(f'RDT model loaded and warmed ({model.model_version}); peak VRAM {_peak_vram_gib():.1f} GiB; ready',
          flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = 'ffa2f4bc2a45e550f63b23a54174f9d38d5fc44a2ef626808bfa54a825d8cfa2'
actual = hashlib.sha256(Path('/content/rdt1b_server.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/rdt1b_server.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/rdt1b_server.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/rdt1b_server.py matches control/vla_bridge/rdt1b_server.py (sha256 {actual[:12]})')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `rdt1b_server.py`, and whatever listens on port 3005.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/rdt1b_server.log`).
It only succeeds once **its own** still-running process owns port 3005.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
import shutil
import sys

sys.path.insert(0, '/content')
from colab_server_utils import start_server, stop_servers, wait_until_ready

PORT = 3005
RDT_LOG = '/content/rdt1b_server.log'
STOP_PATTERNS = ['rdt1b_server.py']
RDT_DIR = '/content/RoboticsDiffusionTransformer'
RDT_PYTHON = '/content/rdt_venv/bin/python'

free_gb = shutil.disk_usage('/content').free / 1e9
if free_gb < 60:
    raise RuntimeError(f'only {free_gb:.0f} GB free on /content; RDT needs ~55 GB (T5-XXL alone is 44.5 GB)')

stop_servers(PORT, STOP_PATTERNS)
rdt_server_process = start_server(
    [RDT_PYTHON, '-u', '/content/rdt1b_server.py', '--port', '3005', '--rdt-repo', RDT_DIR],
    RDT_LOG,
    env_overrides={'TOKENIZERS_PARALLELISM': 'false'},
    cwd=RDT_DIR,
)
health = wait_until_ready(
    rdt_server_process, PORT, RDT_LOG,
    timeout_s=7200, health_url='http://127.0.0.1:3005/health',
)
print('RDT adapter ready:', {k: health['model'][k] for k in (
    'checkpoint', 'revision', 'action_space', 'state_indices', 'control_hz', 'chunk_step', 't5_offload_dir')})
print([line for line in open(RDT_LOG) if 'GiB' in line or 'OutOfMemory' in line])

## Open an HTTP tunnel to port 3005

In [ ]:
from pyngrok import ngrok

rdt_tunnel = ngrok.connect(3005, "http")  # plain JSON-over-HTTP
print(f"RDT endpoint: {rdt_tunnel.public_url}")
SMOKE = ".venv/bin/python -m vla_bridge.backend_smoke --backend rdt1b --server-address https://<rdt_tunnel> --instruction 'pick up the cube'"
EPISODE = '.venv/bin/python run_vla_episode.py --backend rdt1b --server-address https://RDT_TUNNEL.ngrok-free.app --instruction "pick up the cube" --out outputs/rdt1b_UNIQUE_NAME --max-steps 20'
for placeholder in ('https://<rdt_tunnel>', 'https://RDT_TUNNEL.ngrok-free.app'):
    SMOKE = SMOKE.replace(placeholder, rdt_tunnel.public_url)
    EPISODE = EPISODE.replace(placeholder, rdt_tunnel.public_url)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

## Run locally

Then, locally in `control/` (pass the base URL; the client appends `/action`):

```bash
.venv/bin/python run_vla_episode.py --backend rdt1b \
  --server-address https://RDT_TUNNEL.ngrok-free.app \
  --instruction "pick up the cube" \
  --out outputs/rdt1b_UNIQUE_NAME --max-steps 20
```

Each step's `model_version` should read `robotics-diffusion-transformer/rdt-1b@eb09036cc64c`. The action
is an absolute joint target: radians are converted to degrees, and the gripper's [0, 1] to percent.
There is no IK. Every action then goes through the safety validator.

For the alternate, add `--checkpoint robotics-diffusion-transformer/rdt-170m`, the same value the server
was launched with.

A step that logs `http-error-holding-position` means the adapter returned an error. Check
`/content/rdt1b_server.log`:
- 500 `rdt inference failed` means the model produced a non-finite or wrong-shaped action.
- 400 means a bad request. With `--no-text-encoder`, this includes an instruction that has no
  precomputed embedding.

Expect motion that is not task-meaningful. The base checkpoint has never seen SO-101, and the per-step
safety clamp bounds every jump.

Smoke run first (synthetic hardware, the arm cannot move; the tunnel cell prints it with the URL filled in):

```bash
.venv/bin/python -m vla_bridge.backend_smoke --backend rdt1b --server-address https://<rdt_tunnel> --instruction 'pick up the cube'
```

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/rdt1b_server.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `rdt1b_server.py` owns port 3005.
- Package versions, GPU memory, and `/health`.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/rdt1b_server.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(3005, 'rdt1b_server.py', globals().get('rdt_server_process'))

In [ ]:
import subprocess

RDT_PYTHON = '/content/rdt_venv/bin/python'
probe = (
    "import importlib.metadata as m\n"
    "for p in ('torch', 'transformers', 'diffusers', 'timm', 'accelerate', 'huggingface_hub', 'numpy'):\n"
    "    try:\n"
    "        print(p, m.version(p))\n"
    "    except m.PackageNotFoundError:\n"
    "        print(p, 'not installed')\n"
)
result = subprocess.run([RDT_PYTHON, '-c', probe], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import health_report

health_report(3005)

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted server is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 3005 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# RDT teardown -- every session, after the local run has finished.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'rdt_tunnel' in globals():
    try:
        ngrok.disconnect(rdt_tunnel.public_url)
        print(f'Disconnected {rdt_tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('rdt_server_process'), 3005, ['rdt1b_server.py'])